In [74]:
import os
import sys
import numpy as np
import subprocess
from shutil import which

In [75]:
# 0..21 から 21(=index 20) を除くチーム一覧
def get_teams():
    arr = np.arange(1, 23)
    return np.delete(arr, 20)  # 21 をスキップ（コンテスト仕様に合わせる）

In [76]:
def loop_for_all_teams(
    command_template,
    *,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    """
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...] のようなリスト
    dry_run: True -> 実行せず展開コマンドのみ表示
    strict: True -> {id:02d} が一つも無ければ例外
    continue_on_error: True -> 失敗しても次の team へ。False -> そこで中断
    cwd: サブプロセスの作業ディレクトリ（attack/ ディレクトリの相対パス解決に使える）
    """
    id_indices = [i for i, arg in enumerate(command_template)
                  if isinstance(arg, str) and "{id:02d}" in arg]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    # 'python' を実行ファイルに置換（環境ズレ回避）
    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    # 事前: 実行ファイルの存在チェック（python 以外の最初の実体コマンドにも対応）
    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    for team in get_teams():
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        # 簡易プリフライト: 既知の入力系ファイルっぽい引数を存在確認
        # （.csv, .json かつ -o/--out* ではない位置を対象にする）
        def is_out_flag(i):
            return isinstance(cmd[i-1], str) and (
                cmd[i-1] in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or cmd[i-1].startswith("--out")
            )

        missing_inputs = []
        for i, a in enumerate(cmd):
            if isinstance(a, str) and (a.endswith(".csv") or a.endswith(".json")):
                if not is_out_flag(i):  # 出力ではなく入力と推定
                    apath = a if cwd is None else os.path.join(cwd, a)
                    if not os.path.exists(apath):
                        missing_inputs.append(a)

        print(">>", " ".join(cmd))
        if missing_inputs:
            msg = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", msg)
                continue
            else:
                raise FileNotFoundError(msg)

        if dry_run:
            continue

        try:
            # 標準出力・標準エラーを取得して、失敗時に見せる
            completed = subprocess.run(
                cmd, check=True, cwd=cwd,
                capture_output=True, text=True
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as e:
            print(f"\n[ERROR] team {team} command failed with code {e.returncode}")
            if e.stdout:
                print("--- stdout ---")
                print(e.stdout.strip())
            if e.stderr:
                print("--- stderr ---")
                print(e.stderr.strip())
            if not continue_on_error:
                raise

In [ ]:
# 必要なら cwd='プロジェクトのルート' を指定（例: cwd=r'c:\work\pwscup2025'）
# cwd = r"C:\Users\kikus\Documents\統数研\pwscup2025-scripts"  # <- 適宜書き換え
# os.chdir(cwd)

In [77]:
## Attack of Ci with original samples
Ci_attack_original = ["python", "attack/attack_Ci.py", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "-o", "out/C{id:02d}_inferred.csv"]
loop_for_all_teams(Ci_attack_original)
print(f"sample Ci-attack completed")

>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv -o out/C01_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv -o out/C02_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A03.csv out/PWSCUP2025_Pre_Data_for_Attack/C03_fix.csv -o out/C03_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A04.csv out/PWSCUP2025_Pre_Data_for_Attack/C04_fix.csv -o out/C04_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py out/PWSCUP2025_Pre_Data_for_Attack/A0

In [80]:
## Attack of Ci with extended version
Ci_attack_extended = ["python", "attack/attack_Ci_ex.py", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "-o", "out/C{id:02d}_inferred_ex.csv", "-k", "1"]
loop_for_all_teams(Ci_attack_extended)
print(f"extended Ci-attack completed")

>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A01.csv out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv -o out/C01_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A02.csv out/PWSCUP2025_Pre_Data_for_Attack/C02_fix.csv -o out/C02_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A03.csv out/PWSCUP2025_Pre_Data_for_Attack/C03_fix.csv -o out/C03_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py out/PWSCUP2025_Pre_Data_for_Attack/A04.csv out/PWSCUP2025_Pre_Data_for_Attack/C04_fix.csv -o out/C04_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack

In [81]:
## Attack of Ci with k-NN version
mode = "nn"
k = 5 # choose Nearest k neighbors
Ci_attack_knn = ["python", "attack/attack_Ci_ex_greedy.py", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-m", mode, "-k", str(k), "-o", f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv"]
loop_for_all_teams(Ci_attack_knn)
print(f"k-NN Ci-attack completed")

>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex_greedy.py out/PWSCUP2025_Pre_Data_for_Attack/C01_fix.csv out/PWSCUP2025_Pre_Data_for_Attack/A01.csv -m nn -k 5 -o out/C01_inferred_ex_greedy_k5_nn.csv
dists: [[1.34626579 1.75256181 2.14992976 2.16133809 2.17490435]
 [0.84782469 0.91240346 0.99158037 1.00940716 1.02233338]
 [0.57717073 0.67538285 0.70600224 0.77023667 0.79260063]
 ...
 [1.25265384 1.32750666 1.35667264 1.36075032 1.37555325]
 [0.62510413 0.65466034 0.66586661 0.67827851 0.68271232]
 [0.80501175 0.86199027 0.90041375 1.06054902 1.23266959]]
inds: [[43416 72660 45026 10865 22601]
 [44524 85869 34405 52516 47774]
 [88977 86553 67643 36144 48543]
 ...
 [33283 18698  8797 59885 32396]
 [29881  9209  5322 59978 78543]
 [ 9687 98492 21321 85345  7482]]
dists.shape: (10000, 5), inds.shape: (10000, 5)
1.3462657928466797 43416
idx: [43416 72660 45026 ... 21321 85345  7482]
distances: [1.34626579 1.75256181 2.14992976 ... 0.90041375 1.06054902 1.23266959]


In [ ]:
## Attack of Ci with greedy-k-NN version
mode = "greedy"
k = 300 # choose greedy k ranks (need enough big k for computation)
Ci_attack_greedy = ["python", "attack/attack_Ci_ex_greedy.py", "out/PWSCUP2025_Pre_Data_for_Attack/C{id:02d}_fix.csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-m", mode, "-k", str(k), "-o", f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv", "--out-map", f"out/C{{id:02d}}_matchmap_k{k}.csv"]
loop_for_all_teams(Ci_attack_greedy)
print(f"greedy-k-NN Ci-attack completed")

greedy-k-NN Ci-attack completed


In [ ]:
## Attack of Di with original samples
Di_attack_original = ["python", "attack/attack_Di.py", "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv"]
loop_for_all_teams(Di_attack_original)
print(f"sample Di-attack completed")

sample Di-attack completed


In [ ]:
## Attack of Di with extended version
# python attack\attack_Di_ex.py out\PWSCUP2025_Pre_Data_for_Attack\D22.json out\PWSCUP2025_Pre_Data_for_Attack\A22.csv [--pred-threshold 0.5] [--pred-topk 10000] [--pred-pos-ratio 0.10] [--conf-threshold 0.1] [--conf-topk 10000] [--conf-pos-ratio 0.10] --out-pred out/inferred_membership1_22_ex.csv --out-conf out/inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
Di_attack_extended = ["python", "attack/attack_Di_ex.py", "out/PWSCUP2025_Pre_Data_for_Attack/D{id:02d}.json", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "--out-pred", "out/inferred_membership1_{id:02d}_ex.csv", "--out-conf", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Di_attack_extended)
print(f"extended Di-attack completed")

extended Di-attack completed


In [ ]:
## Original Combination Attack on Ci and Di.
# python attack\attack_Di_ex.py out\PWSCUP2025_Pre_Data_for_Attack\D22.json out\PWSCUP2025_Pre_Data_for_Attack\A22.csv [--pred-threshold 0.5] [--pred-topk 10000] [--pred-pos-ratio 0.10] [--conf-threshold 0.1] [--conf-topk 10000] [--conf-pos-ratio 0.10] --out-pred out/inferred_membership1_22_ex.csv --out-conf out/inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
Combi_attack_original = ["python", "attack/attack_example_ex.py", "--Ai_csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-o", "out/Fij_{id:02d}.csv", "out/C{id:02d}_inferred.csv", "out/inferred_membership1_{id:02d}_ex.csv", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Combi_attack_original)
print(f"original combination attack completed")

original combination attack completed


In [ ]:
## Extended Combination Attack on Ci and Di.
# python attack\attack_example_ex.py --Ai_csv out\PWSCUP2025_Pre_Data_for_Attack\A22.csv -o out\Fij_22.csv -l 10000 out\C22_inferred.csv out\inferred_membership1_22_ex.csv out\inferred_membership2_22_ex.csv
# if threshold is not specified, default values will be used.
limit = 10000 # limit of number of samples to be inferred

Combi_attack_extended = ["python", "attack/attack_example_ex.py", "--Ai_csv", "out/PWSCUP2025_Pre_Data_for_Attack/A{id:02d}.csv", "-o", "out/Fij_{id:02d}.csv", "-l", str(limit), f"out/C{{id:02d}}_inferred_ex_greedy_k{k}_{mode}.csv", "out/inferred_membership1_{id:02d}_ex.csv", "out/inferred_membership2_{id:02d}_ex.csv"]
loop_for_all_teams(Combi_attack_extended)
print(f"extended combination attack completed")

extended combination attack completed
